## Preliminares

In [1]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.model_selection import cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import plotly.express as px
from category_encoders import LeaveOneOutEncoder
from src.evaluators import *

In [2]:
# Abrir archivo clean_data
data_folder = "data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17947 entries, 0 to 17946
Data columns (total 36 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   YearsSinceAdded                 17947 non-null  float64 
 1   Monthly_Return                  17947 non-null  float64 
 2   Monthly_Variance                17947 non-null  float64 
 3   Market_Covariance               17947 non-null  float64 
 4   operatingMargins                17947 non-null  float64 
 5   profitMargins                   17947 non-null  float64 
 6   ReturnOnAssets                  17947 non-null  float64 
 7   Revenue_Growth_YoY              17947 non-null  float64 
 8   Revenue_Growth_QoQ              17947 non-null  float64 
 9   EBITDA_Growth_YoY               17947 non-null  float64 
 10  EBITDA_Growth_QoQ               17947 non-null  float64 
 11  FedFundsRate                    17947 non-null  float64 
 12  10YTreasuryYield  

## Feature Engineering

In [3]:
# Sección para crear variables en la fase de modelado, la mayor parte de las features fue creada en pasos anteriores
# Se crea la variable Quarter para modelar estacionalidad trimestral
df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
df['Quarter'] = df['Date'].dt.quarter

# Se convierte a category
df['Quarter'] = df['Quarter'].astype('category')

# Calcular la aceleración neta (Momento - Tendencia) para variables de crecimiento
df['CPI_Acceleration'] = df['CPI_QoQ'] - df['CPI_YoY']
df['M2_Acceleration'] = df['M2_QoQ'] - df['M2_YoY']
df['INDPRO_Acceleration'] = df['INDPRO_QoQ'] - df['INDPRO_YoY']

df['Revenue_Growth_Acceleration'] = df['Revenue_Growth_QoQ'] - df['Revenue_Growth_YoY']
df['EBITDA_Growth_Acceleration'] = df['EBITDA_Growth_QoQ'] - df['EBITDA_Growth_YoY']

# Se eliminan las columnas trimestrales
df = df.drop(columns= ['CPI_QoQ', 'M2_QoQ', 'INDPRO_QoQ', 'Revenue_Growth_QoQ', 'EBITDA_Growth_QoQ'])


In [8]:
df.SubIndustry.value_counts().tail(10)

SubIndustry
IndustrialREITs                      39
Metal,Glass&PlasticContainers        39
WirelessTelecommunicationServices    39
TimberREITs                          39
Computer&ElectronicsRetail           38
FoodRetail                           38
HomefurnishingRetail                 38
Footwear                             37
HeavyElectricalEquipment             15
HealthCareTechnology                 15
Name: count, dtype: int64

# Modelo de ensamblado de árboles RandomForest

In [ ]:
# Ratios de valuación
ratios = ['PriceToBook_Transformed', 'PE_Trailing_Transformed', 'EnterpriseToEbitda_Transformed']

# Se asegura el ordenamiento por fecha
df.sort_values(by='Date', inplace=True)

# Se define la variable objetivo y las variables predictoras
label = 'EnterpriseToEbitda_Transformed'
X = df.drop(columns=ratios + ['Ticker', 'Date']) # Se quitan los ratios de valuación restantes, Ticker y Fecha de las variables predictoras
y = df[label]

# identificar columnas numéricas 
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

# Variables categóricas: se separan en baja y alta cardinalidad para aplicar diferentes técnicas de codificación
low_cardinality_cols = ['Sector', 'Quarter']
high_cardinality_cols = ['SubIndustry']

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('low_card', OneHotEncoder(handle_unknown='ignore'), low_cardinality_cols),
    ('high_card', LeaveOneOutEncoder(random_state=42), high_cardinality_cols)
])

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        max_features='sqrt',
        max_samples= 0.8,
        min_samples_split= 10         
        ))
])

print("Entrenando el modelo con datos completos...")
pipe.fit(X, y)
r2_completo = pipe.score(X, y)
print(f"Entrenamiento finalizado. R2 en datos completos: {r2_completo:.4f}")

Entrenando el modelo con datos completos...
Entrenamiento finalizado. R2 en datos completos: 0.8910


In [10]:
# Test de validación cruzada
# Configurar el validador de series temporales
tscv = TimeSeriesSplit(n_splits=5) # n_splits=5 creará 5 particiones temporales secuenciales

# 3. Test de validación cruzada temporal
cross_val_scores = cross_val_score(
    estimator=pipe, 
    X=X, 
    y=y, 
    cv=tscv,         
    scoring='r2',
    n_jobs=-1        
)

print(f"R² promedio Time Series CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio Time Series CV: 0.6472 ± 0.0562


In [10]:
# Importancia de factores en el modelo
rf_model = pipe.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
44,high_card__SubIndustry_Grouped,0.153432
7,num__Revenue_Growth_YoY,0.084788
27,num__Revenue_Growth_Acceleration,0.075607
6,num__ReturnOnAssets,0.072626
5,num__profitMargins,0.060414
4,num__operatingMargins,0.057386
20,num__MarketCap_log,0.053219
0,num__YearsSinceAdded,0.051341
22,num__debtToEquity_log,0.043499
28,num__EBITDA_Growth_Acceleration,0.041797


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Positive_Bias: si los residuos son mayores o iguales a cero.
* Negative_Bias: si los residuos son menores a cero.

In [11]:
# Se dividen los datos para predecir el valor de la última fecha disponible para cada ticker en el conjunto de test
X_train = X.iloc[:-len(df['Ticker'].unique())]  # Todos menos la última fecha de cada ticker
y_train = y.iloc[:-len(df['Ticker'].unique())]
X_test = X.iloc[-len(df['Ticker'].unique()):]   # Solo la última fecha de cada ticker
y_test = y.iloc[-len(df['Ticker'].unique()):]

# Recuperar el ticker usando el indice de y_test
tickers_test = df.loc[y_test.index, 'Ticker']

# Predicciones en el conjunto de test
y_pred = pipe.predict(X_test)

# Consolidar resultados individuales en un DataFrame
resultados_df = pd.DataFrame({
    'Ticker': tickers_test,
    'Predicho': y_pred,
    'Real': y_test
})

# Calcular el residuo para cada predicción
resultados_df['Residuo'] = resultados_df['Predicho'] - resultados_df['Real']

# Agrupar por ticker
resultados_agrupados = resultados_df.groupby('Ticker')[['Predicho', 'Real', 'Residuo']].mean()

# Generar el Cluster sobre el promedio de los residuos
resultados_agrupados['Cluster'] = ['Positive_Bias' if r >= 0 else 'Negative_Bias' 
                                   for r in resultados_agrupados['Residuo']]

# Visualizar
fig = px.scatter(
    resultados_agrupados, 
    x='Real', 
    y='Predicho', 
    color='Cluster',
    hover_name=resultados_agrupados.index, 
    labels={'Real':'Valor Real Promedio', 'Predicho':'Predicción Promedio', 'Cluster':'Sesgo del Modelo'},
    title='Predicciones vs Reales (Agrupado por Ticker)'
)

# Línea de identidad perfecta
min_val = resultados_agrupados['Real'].min()
max_val = resultados_agrupados['Real'].max()
fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
              line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster a nivel Ticker
over_mask = resultados_agrupados['Cluster'] == 'Positive_Bias'
under_mask = resultados_agrupados['Cluster'] == 'Negative_Bias'

print("\nEstadísticas por cluster (a nivel de Ticker):")
print(f"Overprediction: {over_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[over_mask, 'Residuo'].mean():.4f}")
print(f"Underprediction: {under_mask.sum()} tickers, residuo medio global: {resultados_agrupados.loc[under_mask, 'Residuo'].mean():.4f}")


Estadísticas por cluster (a nivel de Ticker):
Overprediction: 261 tickers, residuo medio global: 0.0618
Underprediction: 191 tickers, residuo medio global: -0.0779


In [12]:
# Ordenar resultados por residuos
resultados_agrupados = resultados_agrupados.sort_values(by='Residuo', ascending=False)
# Positive Biases: Sobrepredicciones seleccionando solo valores positivos para los ratios Reales y Predichos
positive_biases = resultados_agrupados[(resultados_agrupados['Real'] > 0) & (resultados_agrupados['Predicho'] > 0) & (resultados_agrupados['Cluster'] == 'Positive_Bias')]

positive_biases.head(20)

,Predicho,Real,Residuo,Cluster
Ticker,,,,
DELL,0.178221,0.037639,0.140582,Positive_Bias
NOW,0.340585,0.210665,0.129920,Positive_Bias
NVDA,0.338062,0.210084,0.127978,Positive_Bias
APP,0.390186,0.282408,0.107778,Positive_Bias
WDAY,0.131570,0.053381,0.078189,Positive_Bias
APH,0.164056,0.097415,0.066641,Positive_Bias
ISRG,0.359765,0.293818,0.065946,Positive_Bias
SNPS,0.267064,0.205787,0.061277,Positive_Bias
MA,0.063394,0.002848,0.060547,Positive_Bias


In [13]:
# Establer Ticker como columna para exportar resultados
positive_biases.reset_index(inplace=True)
positive_biases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Ticker    21 non-null     object 
 1   Predicho  21 non-null     float64
 2   Real      21 non-null     float64
 3   Residuo   21 non-null     float64
 4   Cluster   21 non-null     object 
dtypes: float64(3), object(2)
memory usage: 972.0+ bytes


In [14]:
# Se exportan los sesgos positivos a un archivo CSV para su análisis posterior 
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

os.makedirs(f"{data_folder}/results", exist_ok=True) # Crear carpeta si no existe

positive_biases.to_csv(f"{data_folder}/results/positive_biases_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimización de hiperparámetros

In [16]:
ejecutar_celda = False

if ejecutar_celda:
    nombre_modelo = "Random Forest"
    print(f"Configurando GridSearchCV para {nombre_modelo}")

    # Pipeline usando el preprocesador específico para Random Forest
    modelo_base = Pipeline(steps=[
        ('preprocesador', preprocessor),
        ('rf', RandomForestRegressor(random_state=42))
    ])

    param_grid = {
        'rf__n_estimators': [200, 300],
        'rf__max_depth': [6, 8, 12],
        'rf__min_samples_leaf': [3, 5, 10],
        'rf__min_samples_split': [10, 15],
        'rf__max_samples': [0.7, 0.8],
        'rf__max_features': ['sqrt']
    }

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='r2',
        cv=tscv,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con datos completos
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X, y)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)